In [ ]:
!pip install -q langchain langgraph groq gradio requests

In [ ]:
import os
import requests
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, END
from groq import Groq

In [ ]:
GROQ_API_KEY = "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"
SERPER_API_KEY = "d592c0f7412d92a66d94708a8a3479bbf28383fa"

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
llama-3.1-8b-instant

In [ ]:
MODEL_NAME = "llama-3.1-8b-instant"

def call_llama(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [ ]:
class ITState(TypedDict):
    ticket: str
    category: str
    priority: str
    research: str
    solution: str
    escalation: str

In [ ]:
def triage_agent(state: ITState):

    prompt = f"""
    You are an IT Helpdesk Triage Agent.

    Analyze the ticket below and return:
    1. Category (Network, Software, Hardware, Access, Security, Other)
    2. Priority (Low, Medium, High, Critical)

    Ticket:
    {state['ticket']}

    Return in format:
    Category: <value>
    Priority: <value>
    """

    response = call_llama(prompt)

    lines = response.split("\n")
    category = "Other"
    priority = "Medium"

    for line in lines:
        if "Category:" in line:
            category = line.split(":")[1].strip()
        if "Priority:" in line:
            priority = line.split(":")[1].strip()

    state["category"] = category
    state["priority"] = priority

    return state

In [ ]:
def web_search(query):
    url = "https://google.serper.dev/search"
    payload = {"q": query}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(url, json=payload, headers=headers)
    results = response.json()

    snippets = []
    for item in results.get("organic", [])[:3]:
        snippets.append(item.get("snippet", ""))

    return " ".join(snippets)

In [ ]:
def research_agent(state: ITState):

    query = f"{state['category']} issue: {state['ticket']} resolution steps enterprise IT"

    research_data = web_search(query)

    state["research"] = research_data
    return state

In [ ]:
def resolution_agent(state: ITState):

    prompt = f"""
    You are a Senior IT Support Engineer.

    Ticket:
    {state['ticket']}

    Category: {state['category']}
    Priority: {state['priority']}

    External Research:
    {state['research']}

    Provide structured output:

    Root Cause:
    Step-by-Step Resolution:
    Preventive Measures:
    Confidence Score (%):

    If issue requires specialist team, write ESCALATE.
    """

    solution = call_llama(prompt)

    state["solution"] = solution

    if "ESCALATE" in solution.upper():
        state["escalation"] = "Yes"
    else:
        state["escalation"] = "No"

    return state

In [ ]:
def escalation_agent(state: ITState):

    if state["escalation"] == "Yes":
        state["solution"] += "\n\n🚨 Escalated to Level 3 Support Team."

    return state

In [ ]:
workflow = StateGraph(ITState)

workflow.add_node("Triage", triage_agent)
workflow.add_node("Research", research_agent)
workflow.add_node("Resolution", resolution_agent)
workflow.add_node("Escalation", escalation_agent)

workflow.set_entry_point("Triage")

workflow.add_edge("Triage", "Research")
workflow.add_edge("Research", "Resolution")
workflow.add_edge("Resolution", "Escalation")
workflow.add_edge("Escalation", END)

app = workflow.compile()

In [ ]:
custom_css = """
body {
    background-color: #f5f7fb;
}
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}
h1 {
    color: #1e3a8a;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🖥️ Enterprise IT Ticket Resolution Agent  
    Powered by Llama 3.1 (Groq) + LangGraph + Serper API  
    """)

    with gr.Row():
        with gr.Column(scale=1):
            ticket_input = gr.Textbox(
                label="Enter IT Ticket Description",
                placeholder="Example: Unable to connect to VPN from remote location...",
                lines=6
            )
            resolve_button = gr.Button("Resolve Ticket", variant="primary")

        with gr.Column(scale=1):
            output_box = gr.Markdown()

    def run_agent(ticket):

        state = {
            "ticket": ticket,
            "category": "",
            "priority": "",
            "research": "",
            "solution": "",
            "escalation": ""
        }

        result = app.invoke(state)

        return f"""
### 📌 Category: {result['category']}
### 🚦 Priority: {result['priority']}

---

### 🛠 Resolution

{result['solution']}
"""

    resolve_button.click(
        run_agent,
        inputs=ticket_input,
        outputs=output_box
    )

demo.launch(debug=True)